# Exercise 4 — Time-Resolved Spectral Analysis: Multitaper Spectrogram

**Neural Data Science (WP7)** · LMU Biology

---

### Overview

A static PSD (Ex7) averages across the entire recording — but neural
oscillations come and go.  A **spectrogram** shows how spectral content
evolves over time, revealing transient events like theta bursts, gamma
episodes, and sharp-wave ripples.

In this exercise you will:
1. Load hippocampal LFP data (same dataset as Ex7)
2. Compute a multitaper spectrogram using a sliding window
3. Explore the time–frequency resolution trade-off
4. Identify transient spectral events in the data

### Learning objectives

- Understand the Heisenberg-style trade-off between time and frequency resolution
- Use `wp7_helpers.spectrogram_multitaper` to compute time-frequency representations
- Visualize spectrograms with appropriate scaling (log power)
- Relate time-varying spectral features to neural activity patterns

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import sys
import os

# Add the course library to the path
sys.path.insert(0, '../../lib')
from wp7_helpers import spectrogram_multitaper

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

### Load the data

Same 16-channel hippocampal LFP recording as Ex7, sampled at **1250 Hz**.
We work with a single channel to keep computation fast.

In [ ]:
data_path = '../data/spectral/ws_data_1shank.mat'
# Data is at ../data/spectral/ relative to this exercise directory

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Data not found at {data_path!r}. Adjust data_path to point to ws_data_1shank.mat"
    )

mat = scipy.io.loadmat(data_path)
lfps = mat['lfps']
fs = 1250  # Hz

n_samples, n_channels = lfps.shape
print(f'LFP shape: {lfps.shape} — {n_channels} channels, {n_samples/fs:.1f} s at {fs} Hz')

ch = 0
x = lfps[:, ch]

---
## Section 1 — Visualize the Raw LFP

Before computing the spectrogram, look at the time-domain signal.
Neural states (active exploration, rest, sleep) produce different
oscillatory patterns — you should see these change over time.

> **Think about it:** Can you see periods of strong rhythmic activity
> vs quieter intervals?  At what timescale do these states change?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Plot the first 10 s of LFP
# ──────────────────────────────────────────────────────
# 1. Extract the first 10 seconds
# 2. Create a time axis in seconds
# 3. Plot with labeled axes
# ──────────────────────────────────────────────────────

raise NotImplementedError("Plot the first 10 seconds of LFP")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

10 seconds at 1250 Hz = 12500 samples. Index with `x[:int(10 * fs)]`.
Create time with `np.arange(n) / fs`.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
seg = x[:int(10 * fs)]
t_seg = np.arange(len(seg)) / fs
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
seg = x[:int(10 * fs)]
t_seg = np.arange(len(seg)) / fs

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_seg, seg, linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (a.u.)')
ax.set_title(f'LFP — Channel {ch}, first 10 s')
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 1
assert len(x) == n_samples, "x should be the full signal for channel 0"
print(f"Checkpoint 1 passed — x has {len(x):,} samples")

---
## Section 2 — Computing the Spectrogram

A spectrogram applies PSD estimation to a **sliding window** that moves
across the signal.  `wp7_helpers.spectrogram_multitaper` returns:
- `freqs`: frequency axis (Hz)
- `times`: center time of each window (s)
- `S`: power matrix of shape `(n_freqs, n_times)`

Display with log-power scaling (`10 * np.log10(S)`) and limit the
frequency axis to 1–200 Hz.

> **Think about it:** The spectrogram has one PSD per window.  If the
> window is 1 s and overlap is 50%, how many windows do you get from
> a 30-second signal?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Compute and display the spectrogram
# ──────────────────────────────────────────────────────
# 1. Call spectrogram_multitaper(x, fs=fs, window_sec=1.0,
#                                overlap=0.5, nw=3)
# 2. Convert power to dB: S_db = 10 * np.log10(S)
#    (clip S to avoid log(0): np.clip(S, 1e-30, None))
# 3. Display with plt.pcolormesh(times, freqs, S_db)
# 4. Limit y-axis to 1–200 Hz, add colorbar
# ──────────────────────────────────────────────────────

raise NotImplementedError("Compute the spectrogram and display it")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

`spectrogram_multitaper` divides the signal into overlapping windows,
computes a multitaper PSD for each, and stacks them.  The result `S`
has shape `(n_freqs, n_times)`.  Use `pcolormesh` (not `imshow`) for
correct axis scaling.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
freqs, times, S = spectrogram_multitaper(x, fs=fs, window_sec=1.0,
                                          overlap=0.5, nw=3)
S_db = 10 * np.log10(np.clip(S, 1e-30, None))
```

To restrict display to 1–200 Hz:
```python
freq_mask = freqs <= 200
ax.pcolormesh(times, freqs[freq_mask], S_db[freq_mask, :], ...)
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
freqs, times, S = spectrogram_multitaper(x, fs=fs, window_sec=1.0,
                                          overlap=0.5, nw=3)
S_db = 10 * np.log10(np.clip(S, 1e-30, None))
freq_mask = freqs <= 200

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.pcolormesh(times, freqs[freq_mask], S_db[freq_mask, :],
                    shading='auto', cmap='viridis')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_ylim(1, 200)
ax.set_title('Multitaper Spectrogram (window=1.0 s, NW=3)')
plt.colorbar(im, ax=ax, label='Power (dB)')
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 2
assert S.ndim == 2, f"S should be 2-D, got {S.ndim}-D"
assert S.shape[0] == len(freqs), "S rows should match freqs"
assert S.shape[1] == len(times), "S columns should match times"
assert np.all(S >= 0), "Power should be non-negative"
print(f"Checkpoint 2 passed — S shape: {S.shape} "
      f"({len(freqs)} freqs x {len(times)} time bins)")

---
## Section 3 — Window Length: The Time–Frequency Trade-Off

The window length is the key parameter.  This is a **Heisenberg-style
uncertainty**: you cannot simultaneously have high resolution in both
time and frequency.

- **Short window** (0.2 s): good time resolution, poor frequency resolution
- **Medium window** (1.0 s): balanced
- **Long window** (4.0 s): good frequency resolution, poor time resolution

> **Think about it:** With a 0.2 s window and NW=3, what is the approximate
> frequency resolution?  (Hint: `Δf ≈ 2·NW / window_sec`.)  Can you resolve
> an 8 Hz theta peak?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Compare spectrograms with different windows
# ──────────────────────────────────────────────────────
# 1. Compute spectrograms with window_sec = 0.2, 1.0, 4.0
# 2. Plot them as subplots (3 rows, 1 column)
# 3. Title each with the window length and frequency resolution
#    Freq resolution ≈ 2 * nw / window_sec
# 4. All plots: 1–200 Hz, log-power (dB), same colormap
# ──────────────────────────────────────────────────────

window_values = [0.2, 1.0, 4.0]

raise NotImplementedError("Compare spectrograms with different window lengths")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

Loop over the window values, call `spectrogram_multitaper` for each,
convert to dB, and plot in a subplot.  The frequency resolution is
approximately `2 * NW / window_sec` Hz — compute this for each window
and include it in the subplot title.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
for ax, win_sec in zip(axes, window_values):
    f, t, Sw = spectrogram_multitaper(x, fs=fs, window_sec=win_sec, overlap=0.5, nw=3)
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
fig, axes = plt.subplots(len(window_values), 1, figsize=(12, 4 * len(window_values)),
                          sharex=True)

for ax, win_sec in zip(axes, window_values):
    f, t, Sw = spectrogram_multitaper(x, fs=fs, window_sec=win_sec, overlap=0.5, nw=3)
    Sw_db = 10 * np.log10(np.clip(Sw, 1e-30, None))
    fmask = f <= 200
    im = ax.pcolormesh(t, f[fmask], Sw_db[fmask, :], shading='auto', cmap='viridis')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_ylim(1, 200)
    freq_res = 2 * 3.0 / win_sec
    ax.set_title(f'window = {win_sec} s  (freq res ≈ {freq_res:.1f} Hz)')
    plt.colorbar(im, ax=ax, label='dB')

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 3
f02, t02, S02 = spectrogram_multitaper(x, fs=fs, window_sec=0.2, overlap=0.5, nw=3)
f40, t40, S40 = spectrogram_multitaper(x, fs=fs, window_sec=4.0, overlap=0.5, nw=3)
assert len(t02) > len(t40), (
    "Short windows should produce more time bins than long windows"
)
assert len(f40) > len(f02), (
    "Long windows should produce more frequency bins than short windows"
)
print(f"Checkpoint 3 passed — 0.2 s: {S02.shape}, 4.0 s: {S40.shape}")

---
## Section 4 — Zooming into Transient Events

Use your 1-second-window spectrogram to zoom into a 20-second region.
Overlay the raw LFP trace on top to relate time-domain waveforms to
spectral content.

> **Think about it:** What neural events would you expect to see in a
> hippocampal recording?  How do theta bursts and sharp-wave ripples
> differ in the spectrogram?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Zoom into a time-frequency region
# ──────────────────────────────────────────────────────
# 1. Re-compute (or reuse) the spectrogram with window=1.0 s
# 2. Create a 2-row figure: top = LFP trace, bottom = spectrogram
# 3. Show a 20-second window (e.g. 10–30 s)
# 4. Use shared x-axis so the traces align
# ──────────────────────────────────────────────────────

raise NotImplementedError("Create a zoomed LFP + spectrogram overlay")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

Use `plt.subplots(2, 1, ...)` with `height_ratios=[1, 3]` so the
spectrogram gets more vertical space.  Use `sharex=True` to link the
time axes.  Restrict the time range with `ax.set_xlim(10, 30)`.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7),
                                height_ratios=[1, 3], sharex=True)
```

For the LFP trace, index the raw signal:
```python
t_raw = np.arange(int(10*fs), int(30*fs)) / fs
ax1.plot(t_raw, x[int(10*fs):int(30*fs)], linewidth=0.3)
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
freqs, times, S = spectrogram_multitaper(x, fs=fs, window_sec=1.0,
                                          overlap=0.5, nw=3)
S_db = 10 * np.log10(np.clip(S, 1e-30, None))
freq_mask = freqs <= 200
time_slice = (times >= 10) & (times <= 30)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[1, 3])

t_raw = np.arange(int(10*fs), int(30*fs)) / fs
ax1.plot(t_raw, x[int(10*fs):int(30*fs)], linewidth=0.3, color='k')
ax1.set_ylabel('Amplitude')
ax1.set_title('LFP + Spectrogram — 10–30 s')
ax1.set_xlim(10, 30)

im = ax2.pcolormesh(times[time_slice], freqs[freq_mask],
                     S_db[np.ix_(freq_mask, time_slice)],
                     shading='auto', cmap='viridis')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Frequency (Hz)')
ax2.set_ylim(1, 200)
ax2.set_xlim(10, 30)
plt.colorbar(im, ax=ax2, label='Power (dB)')
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 4
# Verify the spectrogram was computed
assert 'S' in dir() or 'S' in locals(), "Variable S not found"
assert 'freqs' in dir() or 'freqs' in locals(), "Variable freqs not found"
print(f"Checkpoint 4 passed — spectrogram computed with shape {S.shape}")

---
## Reflection

1. **Window choice:** For clinical EEG analysis (detecting seizures that
   evolve over seconds), would you pick a short or long window?  What about
   for detecting fast gamma bursts (~100 ms duration)?

2. **All channels:** If you computed spectrograms for all 16 channels,
   how would you compare them?  What spatial patterns might you expect
   along the electrode shank?

3. **Beyond Fourier:** The spectrogram assumes the signal is locally
   stationary within each window.  What alternatives exist for signals
   with rapid frequency changes?  (Hint: look up *wavelet transforms*
   or *empirical mode decomposition* — the `emd_tutorial` notebook in
   `sourcedata/moodle/ss24-resources/.../Exercise 08/` is a good starting point.)